TERRAIN - multi-city inductive GCN training on OSM (Kaggle GPU).

Setup required in Kaggle:
- Notebook settings: Accelerator = GPU T4 x2 (or T4), Internet = On.
- Secrets: add `GH_PAT` = GitHub classic token with `repo` scope.
Run cells top to bottom. Outputs: /kaggle/working/model.pth and metrics.json.

In [1]:
!pip install -q osmnx networkx scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 5.2 MB/s eta 0:00:00


In [4]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_PAT")
REPO = f"https://{token}@github.com/DeySag/TERRAIN---ISRO-BAH-26.git"
print("len:", len(token)) 

len: 40


In [5]:
!git config --global --add safe.directory /kaggle/working/terrain
!rm -rf /kaggle/working/terrain
!GIT_TERMINAL_PROMPT=0 git -c credential.helper= clone {REPO} /kaggle/working/terrain
!git -C /kaggle/working/terrain remote set-url origin https://github.com/DeySag/TERRAIN---ISRO-BAH-26.git

Cloning into '/kaggle/working/terrain'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 103 (delta 27), reused 91 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.97 MiB | 29.16 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [6]:
import sys; sys.path.insert(0, "/kaggle/working/terrain")
import config, numpy as np, torch, networkx as nx
from criticality.centrality import compute_centrality, detect_gatekeepers
from criticality.graph_adapter import graph_to_tensor
from criticality.gcn_train import train_gcn_multicity
from criticality.gcn_evaluate import evaluate_gcn, rank_gatekeepers
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "devices", torch.cuda.device_count())

torch 2.10.0+cu128 cuda True devices 2


In [13]:
# Skip this cell on re-runs: it wipes the OSM GraphML cache and forces a refetch.
# Run it ONLY when you want to re-download cities from Overpass.
# !rm -f /kaggle/working/cities/*.graphml

In [14]:
import osmnx as ox
from pathlib import Path
print("osmnx version:", ox.__version__)   # diagnostic — confirm it's 2.x
ox.settings.use_cache = True              # replaces ox.config(use_cache=True)
TRAIN_CITIES = {"chandigarh":"Chandigarh, India","jaipur":"Jaipur, India",
                "varanasi":"Varanasi, India","bhopal":"Bhopal, India",
                "indore":"Indore, India","nagpur":"Nagpur, India"}
HOLDOUT_CITIES = {"bengaluru":"Bengaluru, India","delhi":"Delhi, India",
                  "hyderabad":"Hyderabad, India"}
CITY_DIR = Path("/kaggle/working/cities"); CITY_DIR.mkdir(exist_ok=True)
def build_city_graph(place):
    G = ox.graph_from_place(place, network_type="drive")
    G = ox.convert.to_undirected(G)
    G = nx.Graph(G)
    for _, d in G.nodes(data=True):
        lat, lng = d.get("y", 0.0), d.get("x", 0.0)
        d.clear()                     # drop osmid, highway, geometry, etc.
        d["lat"], d["lng"] = lat, lng
    for _, _, d in G.edges(data=True):
        length = d.get("length", 0.0)
        d.clear()                     # drop shapely geometry + OSM tags
        d["length"] = float(length)
    return nx.convert_node_labels_to_integers(G)
def load_or_fetch(name, place):
    p = CITY_DIR / f"{name}.graphml"
    if p.exists() and p.stat().st_size > 0:
        try:
            G = nx.read_graphml(p)
            G = nx.convert_node_labels_to_integers(G)
            for _, _, d in G.edges(data=True):
                d["length"] = float(d.get("length", 0.0))
            return G
        except Exception:
            print(f"re-fetching {name} (corrupt cache)")
    G = build_city_graph(place)
    tmp = p.with_suffix(".tmp")     # write-then-rename = no partial files
    nx.write_graphml(G, tmp)
    tmp.rename(p)
    return G

osmnx version: 2.1.1


In [15]:
cities = {}
for name, place in {**TRAIN_CITIES, **HOLDOUT_CITIES}.items():
    G = load_or_fetch(name, place)
    cities[name] = (G, compute_centrality(G, k=config.BETWEENNESS_K))
    print(f"{name:12s} n={G.number_of_nodes()} m={G.number_of_edges()}")

chandigarh   n=12148 m=17074
jaipur       n=141090 m=190457
varanasi     n=73176 m=91822
bhopal       n=66500 m=88498
indore       n=66767 m=89813
nagpur       n=39925 m=57567
bengaluru    n=155485 m=204814
delhi        n=184076 m=263085
hyderabad    n=137860 m=185850


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, loss = train_gcn_multicity(cities, list(TRAIN_CITIES.keys()),
                                  device=device, save_path="/kaggle/working/model.pth")
print("final loss:", loss.item())

final loss: 2.742434501647949


In [ ]:
import json
model.eval()
results = {}
for name in HOLDOUT_CITIES:
    G, centrality = cities[name]
    x, edge_index, y_cont, node_ids = graph_to_tensor(G, centrality, target="regression")
    mask = torch.ones_like(y_cont, dtype=torch.bool)
    m = evaluate_gcn(model, x, edge_index, y_cont, mask, continuous=y_cont.numpy())
    topk = [node_ids[i] for i in rank_gatekeepers(model, x, edge_index).tolist()][:20]
    results[name] = {**m, "topk": topk}
    print(name, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in m.items()})
json.dump(results, open("/kaggle/working/metrics.json", "w"), indent=2)

# ---- Baselines on the SAME k=500 labels (defensibility table) ----
from sklearn.linear_model import LogisticRegression
from scipy.stats import spearmanr
from criticality.gcn_evaluate import _coverage_at_k, _auc
pct = config.PSEUDO_LABEL_PERCENTILE

def eval_rank(probs, yb):
    return {"coverage_at_k": _coverage_at_k(probs, yb, pct),
            "auc": float(_auc(probs, yb)),
            "spearman": float(spearmanr(probs, yb).statistic)}

Xt, yt = [], []
for name in TRAIN_CITIES:
    x, _, yb, _ = graph_to_tensor(cities[name][0], cities[name][1], target="binary")
    Xt.append(x.numpy()); yt.append(yb.numpy())
clf = LogisticRegression(max_iter=2000).fit(np.concatenate(Xt), np.concatenate(yt))

for name in HOLDOUT_CITIES:
    G, cent = cities[name]
    x, _, yb, _ = graph_to_tensor(G, cent, target="binary")
    xn, ybn = x.numpy(), yb.numpy()
    base = {
        "degree-only":     eval_rank(xn[:, 0], ybn),
        "clustering-only": eval_rank(xn[:, 1], ybn),
        "kcore-only":      eval_rank(xn[:, 2], ybn),
        "logreg-5feat":    eval_rank(clf.predict_proba(xn)[:, 1], ybn),
    }
    print(name, "baselines", {k: {kk: round(vv, 3) for kk, vv in v.items()} for k, v in base.items()})


In [24]:
import os
print(os.listdir("/kaggle/working"))

['model.pth', '.virtual_documents', 'metrics.json', 'cache', 'cities', 'terrain']


In [17]:
!ls -la /kaggle/working/cities/

total 183840
drwxr-xr-x 2 root root     4096 Aug  6 08:32 .
drwxr-xr-x 6 root root     4096 Aug  6 06:56 ..
-rw-r--r-- 1 root root 32917096 Aug  6 07:51 bengaluru.graphml
-rw-r--r-- 1 root root 14056100 Aug  6 07:30 bhopal.graphml
-rw-r--r-- 1 root root  2612295 Aug  6 06:59 chandigarh.graphml
-rw-r--r-- 1 root root 40881056 Aug  6 08:10 delhi.graphml
-rw-r--r-- 1 root root 29515688 Aug  6 08:32 hyderabad.graphml
-rw-r--r-- 1 root root 14197390 Aug  6 07:37 indore.graphml
-rw-r--r-- 1 root root 30256279 Aug  6 07:07 jaipur.graphml
-rw-r--r-- 1 root root  8805723 Aug  6 07:45 nagpur.graphml
-rw-r--r-- 1 root root 14987491 Aug  6 07:22 varanasi.graphml
